# LightGCL++: Yelp Implementation

This notebook provides a compact reference implementation of LightGCL++ on the Yelp dataset. The model combines graph embeddings, an SVD-based global view, and a semantic view. It also supports four configurations: `baseline`, `no_sem`, `no_ada`, and `full`.

> **Note:** The fallback semantic preprocessing uses item identifiers when real item text is unavailable. Replace the fallback strings with item titles, categories, or descriptions for meaningful semantic features.


In [ ]:
# Environment setup
import os
import subprocess
import sys
import zipfile
from pathlib import Path

import torch

WORK_DIR = Path('/kaggle/working')
REPOSITORY_DIR = WORK_DIR / 'LightGCL'
DATA_DIR = REPOSITORY_DIR / 'data'
YELP_ARCHIVE = DATA_DIR / 'yelp.zip'
YELP_DIR = DATA_DIR / 'yelp'

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'],
    check=True,
)

if not REPOSITORY_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/HKUDS/LightGCL.git', str(REPOSITORY_DIR)],
        check=True,
    )

if YELP_ARCHIVE.exists() and not YELP_DIR.exists():
    with zipfile.ZipFile(YELP_ARCHIVE, 'r') as archive:
        archive.extractall(DATA_DIR)

sys.path.insert(0, str(REPOSITORY_DIR))
os.chdir(REPOSITORY_DIR)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Environment ready. Device: {DEVICE}')


In [ ]:
# Create the semantic preprocessing script
# Note: If you have real Yelp item text/title/category, replace item_strings with those texts.
# The current fallback uses item IDs, so it is a weak semantic proxy.

import os

REPO = "/kaggle/working/LightGCL"
os.makedirs(REPO, exist_ok=True)
os.chdir(REPO)

precompute_code = r'''
import argparse
import os
import pickle

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer


def load_adj(data_dir):
    candidates = [
        ("trnMat.pkl", "pkl"),
        ("trn_mat.pkl", "pkl"),
        ("trnMat.npz", "npz"),
        ("trn_mat.npz", "npz"),
        ("train.txt", "txt"),
    ]

    for fname, fmt in candidates:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            continue

        print(f"Found: {fname} (format={fmt})")

        if fmt == "pkl":
            with open(fpath, "rb") as f:
                adj = pickle.load(f)
        elif fmt == "npz":
            adj = sp.load_npz(fpath)
        else:
            rows, cols = [], []
            n_users, n_items = 0, 0
            with open(fpath, "r") as f:
                for line in f:
                    parts = list(map(int, line.strip().split()))
                    if not parts:
                        continue
                    uid, items = parts[0], parts[1:]
                    n_users = max(n_users, uid + 1)
                    for iid in items:
                        n_items = max(n_items, iid + 1)
                        rows.append(uid)
                        cols.append(iid)
            adj = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(n_users, n_items))

        if not sp.issparse(adj):
            adj = sp.csr_matrix(adj)
        adj = adj.tocsr()
        print(f"Shape: {adj.shape}, nnz={adj.nnz}")
        return adj, adj.shape[0], adj.shape[1]

    raise FileNotFoundError(f"No valid dataset file found in {data_dir}. Files: {os.listdir(data_dir)}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=str, default="yelp", choices=["yelp"])
    parser.add_argument("--data_path", type=str, default="./data")
    parser.add_argument("--model", type=str, default="intfloat/e5-small-v2")
    parser.add_argument("--batch", type=int, default=256)
    parser.add_argument("--force", action="store_true")
    args = parser.parse_args()

    data_dir = os.path.join(args.data_path, args.dataset)
    item_path = os.path.join(data_dir, "item_sem_embeds.pt")
    user_path = os.path.join(data_dir, "user_sem_embeds.pt")

    if os.path.exists(item_path) and os.path.exists(user_path) and not args.force:
        print("Semantic embeddings already exist. Use --force to recompute.")
        print(" item:", item_path)
        print(" user:", user_path)
        return

    print(f"\nDataset: {args.dataset}")
    print(f"Directory: {data_dir}")
    print(f"Files: {os.listdir(data_dir)}")

    adj, n_users, n_items = load_adj(data_dir)

    print(f"\nBuilding item strings for {n_items} items...")
    item_strings = [f"passage: Yelp business item {i}" for i in range(n_items)]

    print(f"\nLoading encoder: {args.model}")
    encoder = SentenceTransformer(args.model)

    print("\nEncoding item embeddings...")
    item_embeds = encoder.encode(
        item_strings,
        batch_size=args.batch,
        show_progress_bar=True,
        normalize_embeddings=True,
        device="cpu",
    )
    item_embeds = torch.FloatTensor(item_embeds)

    print("\nComputing user semantic embeddings by averaging interacted item embeddings...")
    deg = torch.FloatTensor(np.asarray(adj.sum(1)).flatten()).clamp(min=1).unsqueeze(1)
    user_np = (adj @ item_embeds.numpy()) / deg.numpy()
    user_embeds = F.normalize(torch.FloatTensor(user_np), dim=-1)

    torch.save(item_embeds, item_path)
    torch.save(user_embeds, user_path)

    print("\nSaved semantic embeddings:")
    print(f"  item: {item_path}  shape={tuple(item_embeds.shape)}")
    print(f"  user: {user_path}  shape={tuple(user_embeds.shape)}")


if __name__ == "__main__":
    main()
'''

with open(f"{REPO}/precompute_sem_yelp.py", "w", encoding="utf-8") as f:
    f.write(precompute_code)

print("Wrote precompute_sem_yelp.py")


In [ ]:
# Precompute semantic embeddings
import subprocess
import sys

command = [
    sys.executable,
    'precompute_sem_yelp.py',
    '--dataset', 'yelp',
    '--data_path', './data',
    '--batch', '256',
]

subprocess.run(command, check=True)


In [ ]:
# Create the LightGCL++ auxiliary module
# This module makes the ablations clear:
# baseline = BPR + normal SVD contrastive
# no_sem   = BPR + adaptive SVD contrastive
# no_ada   = BPR + normal SVD contrastive + semantic contrastive
# full     = BPR + adaptive SVD contrastive + semantic contrastive

import os

REPO = "/kaggle/working/LightGCL"
os.makedirs(REPO, exist_ok=True)
os.chdir(REPO)

semlightgclpp_code = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F


def info_nce_per_sample(x, y, tau=0.2):
    x = F.normalize(x, dim=-1)
    y = F.normalize(y, dim=-1)

    logits = x @ y.t() / tau
    labels = torch.arange(x.size(0), device=x.device)

    return F.cross_entropy(logits, labels, reduction="none")


def info_nce(x, y, tau=0.2):
    return info_nce_per_sample(x, y, tau).mean()


class LightGCLPPAux(nn.Module):
    def __init__(
        self,
        emb_dim,
        user_sem_raw,
        item_sem_raw,
        tau_svd=0.2,
        tau_sem=0.2,
        gamma=0.5,
        use_adaptive=True,
        use_semantic=True,
    ):
        super().__init__()

        self.emb_dim = emb_dim
        self.tau_svd = tau_svd
        self.tau_sem = tau_sem
        self.gamma = gamma
        self.use_adaptive = use_adaptive
        self.use_semantic = use_semantic

        self.register_buffer("user_sem_raw", user_sem_raw.float())
        self.register_buffer("item_sem_raw", item_sem_raw.float())

        self.user_sem_proj = nn.Linear(user_sem_raw.size(1), emb_dim)
        self.item_sem_proj = nn.Linear(item_sem_raw.size(1), emb_dim)

        self.q_user = nn.Linear(emb_dim, emb_dim)
        self.k_sem = nn.Linear(emb_dim, emb_dim)
        self.v_sem = nn.Linear(emb_dim, emb_dim)

        self.svd_gate_user = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, 1),
            nn.Sigmoid(),
        )
        self.svd_gate_item = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, 1),
            nn.Sigmoid(),
        )

    def build_semantic_view(self, uids, pos_items, h_user_graph):
        user_sem = self.user_sem_proj(self.user_sem_raw[uids])
        item_sem = self.item_sem_proj(self.item_sem_raw[pos_items])

        sem_tokens = torch.stack([user_sem, item_sem], dim=1)

        q = self.q_user(h_user_graph).unsqueeze(1)
        k = self.k_sem(sem_tokens)
        v = self.v_sem(sem_tokens)

        attn_score = (q * k).sum(dim=-1) / (self.emb_dim ** 0.5)
        attn = torch.softmax(attn_score, dim=-1).unsqueeze(-1)

        user_sem_view = (attn * v).sum(dim=1)
        item_sem_view = item_sem

        return user_sem_view, item_sem_view

    def svd_contrastive_loss(self, h_user_graph, h_item_graph, h_user_svd, h_item_svd):
        user_loss = info_nce_per_sample(h_user_graph, h_user_svd, self.tau_svd)
        item_loss = info_nce_per_sample(h_item_graph, h_item_svd, self.tau_svd)

        if self.use_adaptive:
            wu = self.svd_gate_user(torch.cat([h_user_graph, h_user_svd], dim=-1)).squeeze(-1)
            wi = self.svd_gate_item(torch.cat([h_item_graph, h_item_svd], dim=-1)).squeeze(-1)

            # weight range: [1-gamma, 1]
            wu = (1.0 - self.gamma) + self.gamma * wu
            wi = (1.0 - self.gamma) + self.gamma * wi

            user_loss = user_loss * wu
            item_loss = item_loss * wi

        return 0.5 * (user_loss.mean() + item_loss.mean())

    def semantic_contrastive_loss(self, uids, pos_items, h_user_graph, h_item_graph):
        if not self.use_semantic:
            return torch.tensor(0.0, device=h_user_graph.device)

        user_sem_view, item_sem_view = self.build_semantic_view(
            uids, pos_items, h_user_graph
        )

        loss_user = info_nce(h_user_graph, user_sem_view, self.tau_sem)
        loss_item = info_nce(h_item_graph, item_sem_view, self.tau_sem)

        return 0.5 * (loss_user + loss_item)

    def forward(
        self,
        uids,
        pos_items,
        h_user_graph,
        h_item_graph,
        h_user_svd,
        h_item_svd,
    ):
        loss_svd = self.svd_contrastive_loss(
            h_user_graph,
            h_item_graph,
            h_user_svd,
            h_item_svd,
        )

        loss_sem = self.semantic_contrastive_loss(
            uids,
            pos_items,
            h_user_graph,
            h_item_graph,
        )

        return loss_svd, loss_sem
'''

with open(f"{REPO}/semlightgclpp.py", "w", encoding="utf-8") as f:
    f.write(semlightgclpp_code)

print("Wrote semlightgclpp.py")


In [ ]:
# Create the training script

import os

REPO = "/kaggle/working/LightGCL"
os.makedirs(REPO, exist_ok=True)
os.chdir(REPO)

main_sem_code = r'''
import argparse, os, sys, time, pickle, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import scipy.sparse as sp

REPO = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, REPO)

from semlightgclpp import LightGCLPPAux, info_nce


def get_args():
    p = argparse.ArgumentParser("LightGCL++ for Yelp")
    p.add_argument("--dataset",    default="yelp", choices=["yelp"])
    p.add_argument("--data_path",  default="./data")
    p.add_argument("--emb_dim",    type=int,   default=64)
    p.add_argument("--layers",     type=int,   default=2)
    p.add_argument("--q",          type=int,   default=50)
    p.add_argument("--lambda1",    type=float, default=0.05)
    p.add_argument("--lambda2",    type=float, default=0.005)
    p.add_argument("--tau1",       type=float, default=0.2)
    p.add_argument("--tau2",       type=float, default=0.2)
    p.add_argument("--gamma",      type=float, default=0.5)
    p.add_argument("--lr",         type=float, default=1e-3)
    p.add_argument("--decay",      type=float, default=1e-4)
    p.add_argument("--batch_size", type=int,   default=2048)
    p.add_argument("--epochs",     type=int,   default=200)
    p.add_argument("--patience",   type=int,   default=20)
    p.add_argument("--topk",       type=int,   default=20)
    p.add_argument("--ablation",   default="full", choices=["full", "baseline", "no_sem", "no_ada"])
    p.add_argument("--seed",       type=int,   default=2026)
    p.add_argument("--device",     default="cuda")
    return p.parse_args()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_data(data_dir):
    with open(os.path.join(data_dir, "trnMat.pkl"), "rb") as f:
        trn = pickle.load(f).tocsr()
    with open(os.path.join(data_dir, "tstMat.pkl"), "rb") as f:
        tst = pickle.load(f).tocsr()

    n_users, n_items = trn.shape
    train_dict, test_dict = {}, {}

    for u in range(n_users):
        ti = trn.getrow(u).indices.tolist()
        if ti:
            train_dict[int(u)] = set(map(int, ti))

        te = tst.getrow(u).indices.tolist()
        if te:
            test_dict[int(u)] = list(map(int, te))

    print(f"  users={n_users}  items={n_items}  train_nnz={trn.nnz}  test_nnz={tst.nnz}")
    return trn, tst, train_dict, test_dict, n_users, n_items


def build_norm_adj(trn_csr, n_users, n_items, device):
    n = n_users + n_items
    zeros_uu = sp.csr_matrix((n_users, n_users))
    zeros_ii = sp.csr_matrix((n_items, n_items))

    A = sp.bmat([[zeros_uu, trn_csr],
                 [trn_csr.T, zeros_ii]], format="csr")

    d = np.asarray(A.sum(1)).flatten() + 1e-8
    D05 = sp.diags(1.0 / np.sqrt(d))
    A_norm = (D05 @ A @ D05).tocoo()

    idx = torch.LongTensor(np.stack([A_norm.row, A_norm.col]))
    val = torch.FloatTensor(A_norm.data)

    return torch.sparse_coo_tensor(idx, val, (n, n)).coalesce().to(device)


def svd_augment(trn_csr, rank, device):
    import scipy.sparse.linalg as la
    print(f"  Computing SVD at rank={rank}...")
    U, S, Vt = la.svds(trn_csr.astype(np.float32), k=rank)
    return (
        torch.FloatTensor(U.copy()).to(device),
        torch.FloatTensor(S.copy()).to(device),
        torch.FloatTensor(Vt.copy()).to(device)
    )


class LightGCL(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, n_layers, adj_norm, U_svd, S_svd, Vt_svd):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers

        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

        self.adj_norm = adj_norm
        self.register_buffer("U_svd", U_svd)
        self.register_buffer("S_svd", S_svd)
        self.register_buffer("Vt_svd", Vt_svd)

    def forward(self):
        x = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        outs = [x]

        for _ in range(self.n_layers):
            x = torch.sparse.mm(self.adj_norm, x)
            outs.append(x)

        x = torch.stack(outs, dim=1).mean(dim=1)

        h_u = x[:self.n_users]
        h_i = x[self.n_users:]

        tmp = self.Vt_svd @ self.item_emb.weight
        h_tilde_u = F.normalize(
            self.U_svd @ (self.S_svd.unsqueeze(1) * tmp) + self.user_emb.weight,
            dim=-1
        )

        tmp2 = self.U_svd.T @ self.user_emb.weight
        h_tilde_i = F.normalize(
            self.Vt_svd.T @ (self.S_svd.unsqueeze(1) * tmp2) + self.item_emb.weight,
            dim=-1
        )

        return h_u, h_i, h_tilde_u, h_tilde_i


def bpr_loss(h_u, h_i, uids, pos, neg, decay):
    u = h_u[uids]
    pi = h_i[pos]
    ni = h_i[neg]

    loss = -F.logsigmoid((u * pi).sum(-1) - (u * ni).sum(-1)).mean()
    reg = (u.norm(2).pow(2) + pi.norm(2).pow(2) + ni.norm(2).pow(2)) / (2 * len(uids))

    return loss + decay * reg


def sample_negatives(uids_np, train_dict, n_items):
    negs = np.empty(len(uids_np), dtype=np.int64)

    for k, u in enumerate(uids_np):
        user_pos = train_dict.get(int(u), set())

        j = np.random.randint(0, n_items)
        while j in user_pos:
            j = np.random.randint(0, n_items)

        negs[k] = j

    return negs


@torch.no_grad()
def evaluate(model, train_dict, test_dict, n_items, device, topk=20):
    model.eval()

    h_u, h_i, _, _ = model.forward()

    test_users = [u for u in test_dict if len(test_dict[u]) > 0]
    recalls, ndcgs = [], []

    for start in range(0, len(test_users), 512):
        batch = test_users[start:start + 512]
        uids = torch.LongTensor(batch).to(device)

        scores = (h_u[uids] @ h_i.T).cpu().numpy()

        for row, uid in enumerate(batch):
            for iid in train_dict.get(uid, set()):
                scores[row, iid] = -1e9

            top = np.argpartition(scores[row], -topk)[-topk:]
            top = top[np.argsort(-scores[row, top])]

            gt = set(test_dict[uid])
            hits = [1 if p in gt else 0 for p in top]

            recalls.append(sum(hits) / max(len(gt), 1))

            dcg = sum(h / math.log2(r + 2) for r, h in enumerate(hits))
            ideal = sum(1 / math.log2(r + 2) for r in range(min(len(gt), topk)))
            ndcgs.append(dcg / max(ideal, 1e-8))

    model.train()
    return float(np.mean(recalls)), float(np.mean(ndcgs))


def train():
    args = get_args()
    set_seed(args.seed)

    device = torch.device(args.device if torch.cuda.is_available() else "cpu")

    print("=" * 72)
    print(f"LightGCL++ | {args.dataset} | ablation={args.ablation} | seed={args.seed} | {device}")
    print("=" * 72)

    data_dir = os.path.join(args.data_path, args.dataset)

    trn_csr, tst_csr, train_dict, test_dict, n_users, n_items = load_data(data_dir)

    adj_norm = build_norm_adj(trn_csr, n_users, n_items, device)
    U_svd, S_svd, Vt_svd = svd_augment(trn_csr, args.q, device)

    model = LightGCL(
        n_users, n_items,
        args.emb_dim,
        args.layers,
        adj_norm,
        U_svd,
        S_svd,
        Vt_svd
    ).to(device)

    aux = None
    use_semantic = args.ablation in ["full", "no_ada"]
    use_adaptive = args.ablation in ["full", "no_sem"]

    if args.ablation != "baseline":
        item_path = os.path.join(data_dir, "item_sem_embeds.pt")
        user_path = os.path.join(data_dir, "user_sem_embeds.pt")

        if os.path.exists(item_path) and os.path.exists(user_path):
            item_sem = torch.load(item_path, map_location="cpu")
            user_sem = torch.load(user_path, map_location="cpu")

            aux = LightGCLPPAux(
                emb_dim=args.emb_dim,
                user_sem_raw=user_sem,
                item_sem_raw=item_sem,
                tau_svd=args.tau1,
                tau_sem=args.tau2,
                gamma=args.gamma,
                use_adaptive=use_adaptive,
                use_semantic=use_semantic,
            ).to(device)

            print("  LightGCL++ auxiliary module ready")
            print(f"  use_adaptive={use_adaptive}, use_semantic={use_semantic}")
        else:
            print("  Semantic embeddings not found. Fallback to baseline.")
            args.ablation = "baseline"

    params = list(model.parameters())
    if aux is not None:
        params += list(aux.parameters())

    optimizer = optim.Adam(params, lr=args.lr)

    all_u = np.array([u for u, items in train_dict.items() for _ in items], dtype=np.int64)
    all_i = np.array([it for u, items in train_dict.items() for it in items], dtype=np.int64)

    print(f"  Training pairs: {len(all_u)}")
    print("  Negative sampling: seeded and resampled every epoch")
    print("  Loss: BPR + lambda1*SVD_CL + lambda2*SEM_CL")
    print(f"  lambda1={args.lambda1}, lambda2={args.lambda2}, gamma={args.gamma}")

    best_r = 0.0
    best_n = 0.0
    patience = 0

    for epoch in range(1, args.epochs + 1):
        model.train()
        if aux is not None:
            aux.train()

        t0 = time.time()
        perm = np.random.permutation(len(all_u))

        ep_loss = 0.0
        n_batch = 0

        for s in range(0, len(all_u), args.batch_size):
            idx = perm[s:s + args.batch_size]

            batch_u_np = all_u[idx]
            batch_i_np = all_i[idx]
            batch_neg_np = sample_negatives(batch_u_np, train_dict, n_items)

            uids = torch.LongTensor(batch_u_np).to(device)
            pos = torch.LongTensor(batch_i_np).to(device)
            neg = torch.LongTensor(batch_neg_np).to(device)

            h_u, h_i, h_tu, h_ti = model.forward()

            loss_bpr = bpr_loss(h_u, h_i, uids, pos, neg, args.decay)
            loss = loss_bpr

            if args.ablation == "baseline":
                loss_svd = 0.5 * (
                    info_nce(h_u[uids], h_tu[uids], args.tau1)
                    + info_nce(h_i[pos], h_ti[pos], args.tau1)
                )
                loss_sem = torch.tensor(0.0, device=device)
                loss = loss + args.lambda1 * loss_svd

            else:
                loss_svd, loss_sem = aux(
                    uids=uids,
                    pos_items=pos,
                    h_user_graph=h_u[uids],
                    h_item_graph=h_i[pos],
                    h_user_svd=h_tu[uids],
                    h_item_svd=h_ti[pos],
                )

                loss = loss + args.lambda1 * loss_svd + args.lambda2 * loss_sem

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            ep_loss += loss.item()
            n_batch += 1

        r, n = evaluate(model, train_dict, test_dict, n_items, device, args.topk)

        elapsed = int(time.time() - t0)

        print(
            f"Ep {epoch:3d}  loss={ep_loss/n_batch:.4f}  "
            f"R@{args.topk}={r:.4f}  N@{args.topk}={n:.4f}  ({elapsed}s)",
            flush=True
        )

        if r > best_r:
            best_r = r
            best_n = n
            patience = 0

            save_obj = {"model": model.state_dict()}
            if aux is not None:
                save_obj["aux"] = aux.state_dict()
            torch.save(save_obj, f"{args.dataset}_{args.ablation}_best.pt")

            print(
                f"  New best  R@{args.topk}={best_r:.4f}  "
                f"N@{args.topk}={best_n:.4f}",
                flush=True
            )
        else:
            patience += 1

            if patience >= args.patience:
                print(f"  Early stop at epoch {epoch}", flush=True)
                break

    print(f"FINAL {args.dataset} [{args.ablation}]  R@{args.topk}={best_r:.4f}  N@{args.topk}={best_n:.4f}", flush=True)

    return best_r, best_n


if __name__ == "__main__":
    train()
'''

with open(f"{REPO}/main_sem_yelp.py", "w", encoding="utf-8") as f:
    f.write(main_sem_code)

print("Wrote main_sem_yelp.py")


In [ ]:
# Run one configuration
import subprocess
import sys

ABLATION = 'full'  # full, baseline, no_sem, or no_ada

command = [
    sys.executable,
    'main_sem_yelp.py',
    '--dataset=yelp',
    '--data_path=./data',
    '--emb_dim=64',
    '--layers=2',
    '--q=50',
    '--lambda1=0.05',
    '--lambda2=0.005',
    '--tau1=0.2',
    '--tau2=0.2',
    '--gamma=0.5',
    '--lr=1e-3',
    '--decay=1e-4',
    '--batch_size=2048',
    '--epochs=1000',
    '--patience=30',
    '--topk=20',
    '--seed=2026',
    f'--ablation={ABLATION}',
    '--device=cuda',
]

subprocess.run(command, check=True)


In [ ]:
# Run the ablation study
import re
import subprocess
import sys

import pandas as pd

CONFIGURATIONS = ['baseline', 'no_sem', 'no_ada', 'full']

COMMON_ARGUMENTS = [
    sys.executable,
    'main_sem_yelp.py',
    '--dataset=yelp',
    '--data_path=./data',
    '--emb_dim=64',
    '--layers=2',
    '--q=50',
    '--lambda1=0.05',
    '--lambda2=0.005',
    '--tau1=0.2',
    '--tau2=0.2',
    '--gamma=0.5',
    '--lr=1e-3',
    '--decay=1e-4',
    '--batch_size=2048',
    '--epochs=1000',
    '--patience=30',
    '--topk=20',
    '--seed=2026',
    '--device=cuda',
]

pattern = re.compile(
    r'FINAL\s+yelp\s+\[(.*?)\]\s+R@20=([\d.]+)\s+N@20=([\d.]+)'
)
records = []

for configuration in CONFIGURATIONS:
    process = subprocess.run(
        COMMON_ARGUMENTS + [f'--ablation={configuration}'],
        capture_output=True,
        check=True,
        text=True,
    )

    match = pattern.search(process.stdout)
    if match is None:
        raise RuntimeError(f'Unable to parse results for {configuration}.')

    name, recall, ndcg = match.groups()
    records.append(
        {'Configuration': name, 'Recall@20': float(recall), 'NDCG@20': float(ndcg)}
    )

results = pd.DataFrame(records).set_index('Configuration')
results
